In [1]:
# ============================================================
# GOVERNANCE SECTION GENERATOR
# IFRS S1/S2 governance | Azure OpenAI REST endpoint
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
# In notebooks, .env is often not loaded because the kernel runs from a different folder.
# find_dotenv() makes the notebook search upward from the current path.
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_URL = os.getenv("AZURE_OPENAI_CHAT_URL")

if AZURE_OPENAI_CHAT_URL:
    AZURE_OPENAI_CHAT_URL = AZURE_OPENAI_CHAT_URL.strip().strip('"').strip("'")

if not AZURE_OPENAI_API_KEY:
    raise ValueError("Missing AZURE_OPENAI_API_KEY. Add it to .env or environment variables.")

if not AZURE_OPENAI_CHAT_URL:
    raise ValueError(
        "Missing AZURE_OPENAI_CHAT_URL. Expected full Azure URL, e.g.\n"
        "https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=2024-02-01"
    )

if not AZURE_OPENAI_CHAT_URL.startswith("https://"):
    raise ValueError(f"Invalid Azure URL: {AZURE_OPENAI_CHAT_URL!r}")

print("Azure OpenAI REST config loaded")
print("URL loaded:", AZURE_OPENAI_CHAT_URL[:80] + "...")
print("Key loaded:", True, "| length:", len(AZURE_OPENAI_API_KEY))


def _azure_chat_completion(messages: list[dict], temperature: float, max_tokens: int, json_mode: bool = False) -> dict:
    """Minimal Azure OpenAI REST call using the full URL stored in AZURE_OPENAI_CHAT_URL."""
    payload = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    req = urllib.request.Request(
        AZURE_OPENAI_CHAT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY,
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        print("Azure HTTP error:", e.code)
        print(body[:1500])
        raise
    except urllib.error.URLError as e:
        print("Azure URL / connection error:", e)
        print("URL used:", repr(AZURE_OPENAI_CHAT_URL))
        raise


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=2200,
        json_mode=False,
    )
    return data["choices"][0]["message"]["content"]


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=1600,
        json_mode=True,
    )
    return json.loads(data["choices"][0]["message"]["content"])

print("LLM helper functions ready")


Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Azure OpenAI REST config loaded
URL loaded: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-4o-...
Key loaded: True | length: 32
LLM helper functions ready


In [2]:
# ── LOAD PAYLOAD ─────────────────────────────────────────────
# Works both locally and in this sandbox if the payload is placed next to the notebook.

candidate_paths = [
    Path("Data/payload_BANK01.json"),
    Path("payload_BANK01.json"),
    Path.cwd() / "Data" / "payload_BANK01.json",
    Path.cwd() / "payload_BANK01.json",
]

PAYLOAD_PATH = next((p for p in candidate_paths if p.exists()), None)
if PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01.json. Put it in Data/payload_BANK01.json "
        "or in the same folder as the notebook."
    )

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

bank_name = payload["bank"]["bank_name"]
print(f"Loaded payload for: {bank_name}")
print(f"Payload path: {PAYLOAD_PATH}")


Loaded payload for: Eurolux Universal Bank AG
Payload path: payload_BANK01.json


In [3]:
# ── EVIDENCE EXTRACTOR ───────────────────────────────────────
# Pulls only the governance-relevant fields from the payload.
# Fixes added:
# - no unsupported committee-evolution story
# - management process evidence from climate_risk_register
# - meeting_id retained for board decision traceability


def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """Summarise the process evidence needed for IFRS S2 §6(b)."""
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    if not risks:
        return {
            "risk_register_available": False,
            "message": "No climate_risk_register records available for the reporting year."
        }

    frequencies = sorted({str(r.get("monitoring_frequency")) for r in risks if _is_present(r.get("monitoring_frequency"))})
    risk_categories = sorted({str(r.get("risk_category")) for r in risks if _is_present(r.get("risk_category"))})
    risk_ratings = sorted({str(r.get("risk_rating")) for r in risks if _is_present(r.get("risk_rating"))})
    scenario_links = sorted({str(r.get("scenario_analysis_link")) for r in risks if _is_present(r.get("scenario_analysis_link"))})
    mitigation_actions = sorted({str(r.get("mitigation_actions")) for r in risks if _is_present(r.get("mitigation_actions"))})

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    # Keep only the most useful examples for prompt compactness.
    material_risk_examples = []
    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0)
        ),
        reverse=True,
    )
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "reporting_year": year,
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "process_summary_instruction": (
            "For IFRS S2 §6(b), describe the management process using these fields: "
            "risk register, monitoring_frequency, risk_category, risk_rating, scenario_analysis_link, "
            "mitigation_actions, erm_integrated_flag, and changed_since_prior_period. Do not invent an "
            "escalation threshold unless explicitly provided; if needed, say significant/material issues are escalated."
        )
    }


def extract_governance_evidence(payload: dict) -> dict:

    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "scenario", "transition", "net_zero", "target", "carbon_credit",
        "remuneration", "tcfd", "green_finance", "physical_risk", "risk", "esg"
    ]

    def decision_score(m: dict) -> int:
        topics = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        ifrs = str(m.get("ifrs_s2_para_evidence", "")).lower()
        score = sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)
        if "6(a)(v)" in ifrs:
            score += 2
        if str(m.get("committee_type", "")).lower() == "full_board":
            score += 1
        return score

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == 2024
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Deduplicate by decision text only; keep the highest-scoring exact record and retain meeting_id.
    best_by_decision = {}
    for m in minutes_2024:
        decision_text = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        current = best_by_decision.get(decision_text)
        if current is None or decision_score(m) > decision_score(current):
            best_by_decision[decision_text] = m

    selected_decisions = []
    for m in sorted(best_by_decision.values(), key=decision_score, reverse=True)[:6]:
        selected_decisions.append({
            "meeting_id": m.get("meeting_id"),
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
            "internal_ref": f"[REF:{m.get('meeting_id')}]",
        })

    gov_2024 = gov_by_year.get("2024", {})

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": extract_management_process_evidence(payload, year=2024),
        "board_decisions_2024": selected_decisions,
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year "
                "where climate-related topics appeared on the agenda. It does NOT mean "
                "percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that "
                "one committee evolved into, replaced, or was renamed as another. State the 2024 "
                "committee name and, if comparative names are used, present them neutrally."
            ),
            "board_decision_traceability": (
                "Each selected decision includes a meeting_id for audit traceability. Meeting IDs "
                "may be used internally but should not be printed in the final report unless required."
            )
        }
    }


evidence = extract_governance_evidence(payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Risk-register records for §6(b): {evidence['management_process_evidence'].get('risk_count')}")
print("Selected decisions with internal refs:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d['date']} | {d['committee']} | {d['decision']} | {d['internal_ref']}")


Evidence extracted for: Eurolux Universal Bank AG
Board decisions selected: 5
Trend years: [2022, 2023, 2024]
Risk-register records for §6(b): 8
Selected decisions with internal refs:
- 2024-01-24 | Full Board | Approved 2024 ESG report for publication | [REF:MTG-BANK01-FB-2024-001]
- 2024-04-15 | ESG & Sustainability Committee | Approved climate scenario analysis methodology | [REF:MTG-BANK01-ESG-2024-011]
- 2024-04-15 | ESG & Sustainability Committee | Approved carbon credit procurement budget | [REF:MTG-BANK01-ESG-2024-010]
- 2024-10-07 | Full Board | Endorsed updated transition plan | [REF:MTG-BANK01-FB-2024-004]
- 2024-11-01 | Full Board | Endorsed net-zero interim target revision | [REF:MTG-BANK01-FB-2024-007]


In [4]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [5]:
# ── IFRS GOVERNANCE REQUIREMENTS ────────────────────────────
# The previous version incorrectly treated IFRS S2 §8-9 as governance paragraph tags.
# In this notebook, §6(a), §6(b), and §7 are the core IFRS S2 governance anchors.
# Skills are kept as governance evidence but tagged more cautiously.

IFRS_GOVERNANCE_REQUIREMENTS = """
IFRS S2 / IFRS S1 GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

IFRS S2 §6(a) — Governance body / board oversight:
  - How the board or board committee oversees climate-related risks and opportunities.
  - How the board is informed about climate-related matters.
  - How climate is considered in oversight of strategy, major transactions, risk management,
    metrics and targets.
  - Whether specific board or committee climate decisions were made during the reporting year.

IFRS S2 §6(b) — Management role:
  - Which management body or role is responsible for climate-related risks and opportunities.
  - The process by which management monitors, assesses and manages climate risks.
  - The inputs used in that process, such as the climate risk register, scenario links,
    monitoring frequency, risk ratings and mitigation actions.
  - How significant or material matters are escalated to the board or board committee.

IFRS S2 §7 — Remuneration:
  - Whether and how climate-related performance metrics are incorporated into remuneration.
  - Percentage of CEO and executive remuneration linked to ESG/climate metrics where available.

Climate skills and competencies:
  - Use as governance evidence under IFRS S2 §6(a) and IFRS S1 governance context.
  - Do not label this subsection as IFRS S2 §9 because that is not a governance paragraph.

External assurance and controls:
  - Use as control evidence supporting governance over reported metrics.
  - Do not label this subsection as IFRS S2 §8.
"""


In [6]:
# ── WRITER SYSTEM PROMPT ─────────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section
of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure language suitable for publication.
- Specific and data-driven — cite exact figures, dates, and percentages.
- Every quantitative claim must come from the provided evidence — never invent numbers.
- Use paragraph references carefully. Do not label climate skills as IFRS S2 §9, and do not label assurance as IFRS S2 §8.
- No vague language such as "demonstrates commitment" unless backed by concrete evidence.
- Avoid overly strong assurance language such as "ensuring"; prefer "supporting", "providing", or "helping".
- Do not add a generic limitation disclaimer at the end.
- Do not hedge when data is clearly available.

HALLUCINATION CONTROL:
- Do not infer that a committee evolved, was renamed, replaced, strengthened, or specialized across years unless the evidence explicitly says so.
- If committee names differ by year, state only that different names are recorded across the comparative period.
- Do not create a transformation narrative from time-series values.
- Use board decision dates only from board_decisions_2024 and keep the date-decision-committee pairing exactly as provided.

OUTPUT FORMAT:
Return markdown with exactly this subsection structure:

### Governance

#### Board oversight [IFRS S2 §6(a)]
...content...

#### Management responsibility [IFRS S2 §6(b)]
...content...

#### Climate skills and competencies [IFRS S2 §6(a); IFRS S1 governance]
...content...

#### Remuneration and climate incentives [IFRS S2 §7]
...content...

#### Board and committee decisions during 2024 [IFRS S2 §6(a)(v)]
...content...

#### External assurance and controls
...content...
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    is_revision = judge_feedback is not None

    base_instructions = f"""
BANK: {evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

GOVERNANCE REQUIREMENTS FOR THIS SECTION:
{IFRS_GOVERNANCE_REQUIREMENTS}

EVIDENCE (use ONLY this data):
{json.dumps(evidence, indent=2, ensure_ascii=False)}

CRITICAL INTERPRETATION RULES:
1. climate_on_board_agenda_pct = percentage of board MEETINGS where climate was on the agenda.
   NOT percentage of agenda time. Write it as: "climate featured on the agenda of X% of board meetings".
2. Committee names are evidence values by year only. Do NOT say "evolved from", "renamed from",
   "replaced", "progressively strengthened", or "specialized" unless explicit evidence says so.
   For BANK01, write: "In 2024, management-level climate governance is led by the Climate Risk Management Committee."
   If mentioning prior years, say only: "The recorded management committee name was X in 2022 and Y in 2023."
3. For IFRS S2 §6(b), include management process evidence from management_process_evidence:
   risk register availability, monitoring frequencies, risk ratings/categories, scenario links,
   mitigation actions, ERM integration and escalation of significant/material matters.
4. Include year-on-year trends for: board climate expertise %, CEO ESG compensation %,
   ESG committee meeting frequency, climate on board agenda %.
5. Cite at least 4 specific board/committee decisions from board_decisions_2024 with dates.
   Use the exact date, committee, and decision pairing provided. Meeting IDs are internal traceability refs;
   do not print them in the final report unless the prompt explicitly asks.
6. For remuneration: cite both CEO ESG compensation % AND all-executive climate remuneration %.
   Note year-on-year movement for both when available.
7. Explain limited assurance safely: limited assurance provides a lower level of assurance than reasonable assurance,
   based on procedures performed over the stated assurance scope. Do NOT say the assurer concluded that no material
   misstatements exist unless that exact assurance conclusion is provided.
"""

    if is_revision:
        return f"""
{base_instructions}

JUDGE FEEDBACK TO ADDRESS IN THIS REVISION:
{judge_feedback}

REVISION RULES:
- Fix every issue the judge flagged.
- Do not remove content that was not criticised.
- Do not add information not present in the evidence.
- Preserve the required subsection structure and corrected paragraph labels.

Write the revised governance section now.
""".strip()

    return f"""
{base_instructions}

Write the complete governance section now.
Follow the exact subsection structure specified in your instructions.
""".strip()


In [7]:
# ── JUDGE SYSTEM PROMPT ──────────────────────────────────────
JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 compliance reviewer and ESG audit specialist.
Your job is to identify genuine gaps in a governance disclosure section — not to reward fluent writing.

SCORING ANCHOR:
  10: Perfect. Every requirement met, every figure cited, all trends present, board decisions specific and traceable, no unsupported interpretation.
  8-9: Strong. All subsections present, minor gap in one area only.
  6-7: Adequate. All subsections present but 2-3 content requirements thin or missing.
  4-5: Weak. Missing subsections or significant content gaps.
  1-3: Fails minimum disclosure requirements.

A score of 9 or 10 requires ALL of the following to be true:
- All 6 subsections present with substantive content.
- climate_on_board_agenda_pct correctly interpreted as meeting frequency, not agenda time.
- At least 4 distinct board/committee decisions cited with exact dates.
- Year-on-year trends present for board expertise, CEO compensation, ESG committee meetings and climate agenda frequency.
- Both CEO ESG % and all-executive climate % cited in remuneration.
- IFRS S2 §6(b) management responsibility includes process evidence: risk register/ERM integration, monitoring frequency, risk categories or ratings, scenario links or mitigation actions, and escalation wording.
- No unsupported committee evolution / renaming / strengthening narrative.
- Climate skills are NOT tagged as IFRS S2 §9.
- External assurance is NOT tagged as IFRS S2 §8.
- Limited assurance is explained safely as lower assurance than reasonable assurance.
- No generic limitation disclaimer at the end.
- No unsupported claims such as "ensures", "guarantees", "fully aligned", or "fully resilient".

Score cannot exceed 8 if any of the above is false.
Score cannot exceed 6 if any required subsection is missing.
You must return valid JSON only — no other text.
""".strip()


def build_judge_prompt(draft: str, evidence: dict) -> str:

    gov_2024 = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    decisions = evidence.get("board_decisions_2024", [])
    management_process = evidence.get("management_process_evidence", {})

    return f"""
Evaluate this governance section draft against the governance requirements.

DRAFT TO EVALUATE:
{draft}

KEY DATA AVAILABLE TO THE WRITER:
- Board size: {gov_2024.get('board_size')} members
- Independent directors: {gov_2024.get('independent_directors_pct')}%
- ESG committee meetings 2024: {gov_2024.get('esg_committee_meetings_per_year')}
- Board climate expertise: 2022={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('board_climate_expertise_pct')}%
- CEO ESG compensation: 2022={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('ceo_esg_compensation_pct')}%
- All-exec climate remuneration 2024: {gov_2024.get('all_exec_climate_remuneration_pct')}%
- Climate on board agenda: 2022={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('climate_on_board_agenda_pct')}%
- Management committee 2024: {gov_2024.get('management_committee_name')}
- Management process evidence: {json.dumps(management_process, ensure_ascii=False)}
- External assurance: {gov_2024.get('external_assurance')} by {gov_2024.get('assurance_provider')} under {gov_2024.get('assurance_standard')}; scope={gov_2024.get('assurance_scope')}
- Available board decisions with meeting IDs: {json.dumps(decisions, ensure_ascii=False)}

VERIFICATION CHECKLIST — answer each with true/false:
1. all_six_subsections_present: Are all 6 required subsections present?
2. agenda_pct_correct: Is climate_on_board_agenda_pct described as meeting frequency, not agenda time?
3. four_distinct_decisions: Are at least 4 distinct board/committee decisions cited with exact dates?
4. yoy_trends_present: Are year-on-year trends present for expertise, CEO compensation, ESG committee meetings, and climate agenda frequency?
5. both_remuneration_figures: Are both CEO ESG % and all-exec climate % cited?
6. s2_6b_process_complete: Does management responsibility include process evidence, not just committee name?
7. no_unsupported_committee_evolution: No claim that committees evolved/renamed/replaced/strengthened unless explicitly evidenced?
8. correct_ifrs_tags: Skills not tagged as IFRS S2 §9 and assurance not tagged as IFRS S2 §8?
9. assurance_explained_safely: Limited assurance explained without overstating the assurance conclusion?
10. no_limitation_disclaimer: No generic limitation disclaimer at the end?
11. no_unsupported_strong_claims: No unsupported words like ensures/guarantees/fully resilient/fully aligned?

COUNT how many checklist items are false.
Apply score ceiling:
- 0 false: score can reach 9-10
- 1 false: score cannot exceed 8
- 2 false: score cannot exceed 7
- 3+ false: score cannot exceed 6

Return this exact JSON structure:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approved": <true if overall_score >= 8 and 0 false checklist items, else false>,
  "checklist": {{
    "all_six_subsections_present": <true/false>,
    "agenda_pct_correct": <true/false>,
    "four_distinct_decisions": <true/false>,
    "yoy_trends_present": <true/false>,
    "both_remuneration_figures": <true/false>,
    "s2_6b_process_complete": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "correct_ifrs_tags": <true/false>,
    "assurance_explained_safely": <true/false>,
    "no_limitation_disclaimer": <true/false>,
    "no_unsupported_strong_claims": <true/false>,
    "false_count": <integer>
  }},
  "main_issues": [<list of specific issues found>],
  "required_fixes": [<specific actionable instructions for the reviser>]
}}
""".strip()


In [8]:
# ── DETERMINISTIC RULE CHECKS ────────────────────────────────
# These run before the LLM judge and can override it.
# They catch the known failure modes from the governance evaluation.


def rule_check(draft: str) -> dict:
    text = draft.lower()

    required_subsections = [
        "#### board oversight",
        "#### management responsibility",
        "#### climate skills and competencies",
        "#### remuneration and climate incentives",
        "#### board and committee decisions",
        "#### external assurance",
    ]

    missing_subsections = [s for s in required_subsections if s not in text]

    unsupported_evolution_patterns = [
        "evolved from",
        "evolved into",
        "progressive strengthening",
        "progressively strengthening",
        "specialization of the bank",
        "specialisation of the bank",
        "was renamed",
        "renamed as",
        "replaced by",
        "transformed into",
    ]

    hard_fails = {
        "agenda_time_misinterpretation": (
            "agenda time" in text or
            "% of the board's agenda" in text or
            "dedicated to climate" in text
        ),
        "limitation_disclaimer": any(
            phrase in text for phrase in [
                "we acknowledge this limitation",
                "absence of prepared evidence",
                "unable to provide",
                "evidence is limited",
                "will strive to provide",
                "this section acknowledges",
            ]
        ),
        "unsupported_committee_evolution": any(p in text for p in unsupported_evolution_patterns),
        "wrong_skills_ifrs_tag": "climate skills and competencies [ifrs s2 §9]" in text,
        "wrong_assurance_ifrs_tag": "external assurance and controls [ifrs s2 §8]" in text,
        "overstrong_assurance_or_control_language": any(
            phrase in text for phrase in [
                "ensuring regular updates",
                "ensuring transparency",
                "ensuring the reliability",
                "guarantees",
                "fully resilient",
                "fully aligned",
            ]
        ),
        "missing_ifrs_references": ("§6" not in draft and "§7" not in draft),
    }

    structure_ok = len(missing_subsections) == 0
    content_ok = not any(hard_fails.values())

    return {
        "passed": structure_ok and content_ok,
        "structure_ok": structure_ok,
        "content_ok": content_ok,
        "missing_subsections": missing_subsections,
        "hard_fails": {k: v for k, v in hard_fails.items() if v},
        "required_fixes": (
            [f"Add missing subsection: {s}" for s in missing_subsections] +
            [f"Fix hard fail: {k}" for k, v in hard_fails.items() if v]
        )
    }


In [9]:
# ── LANGGRAPH NODES ──────────────────────────────────────────

def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
        temperature=0.2
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]

    # Step 1: deterministic rule checks
    rules = rule_check(draft)
    print(f"\nRule check: {'PASSED' if rules['passed'] else 'FAILED'}")
    if not rules["passed"]:
        print(f"  Issues: {rules['required_fixes']}")

    if not rules["passed"]:
        # Force a structured judge result from rule failures
        judge_result = {
            "overall_score": 4 if rules["structure_ok"] else 3,
            "evidence_support_score": 5,
            "ifrs_alignment_score": 4,
            "specificity_score": 5,
            "hallucination_risk": "medium",
            "approved": False,
            "checklist": {
                "all_six_subsections_present": rules["structure_ok"],
                "false_count": len(rules["required_fixes"])
            },
            "main_issues": rules["required_fixes"],
            "required_fixes": rules["required_fixes"],
            "rule_check_override": True
        }
        return {
            **state,
            "judge_result": judge_result,
            "status": "judging"
        }

    # Step 2: LLM judge
    judge_prompt = build_judge_prompt(draft, state["evidence"])
    judge_result = call_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt
    )

    # Step 3: hard score ceiling enforcement
    false_count = judge_result.get("checklist", {}).get("false_count", 0)
    score = judge_result.get("overall_score", 0)

    ceilings = {0: 10, 1: 8, 2: 7}
    ceiling = ceilings.get(false_count, 6)
    if score > ceiling:
        judge_result["overall_score"] = ceiling
        judge_result["score_ceiling_applied"] = f"Capped at {ceiling} due to {false_count} failed checks"

    # Step 4: approved only if score >= 8 AND false_count == 0
    judge_result["approved"] = (
        judge_result.get("overall_score", 0) >= 8 and
        false_count == 0
    )

    print(f"\nJudge result:")
    print(f"  Score: {judge_result.get('overall_score')}/10")
    print(f"  Approved: {judge_result.get('approved')}")
    print(f"  False checks: {false_count}")
    if judge_result.get("main_issues"):
        print(f"  Issues: {judge_result['main_issues']}")

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging"
    }


def reviser_node(state: GovernanceState) -> GovernanceState:
    return {
        **state,
        "revision_count": state["revision_count"] + 1,
        "status": "drafting"
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


# ── ROUTING ──────────────────────────────────────────────────
def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [10]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "writer")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

Graph compiled


In [11]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

Starting governance generation for: Eurolux Universal Bank AG


WRITER (initial draft)
Draft length: 675 words

Rule check: PASSED

Judge result:
  Score: 8/10
  Approved: False
  False checks: 1
  Issues: ["Unsupported narrative about committee name changes: The draft states the management committee was named 'Group Sustainability Committee' in 2022 and 'ESG Executive Committee' in 2023, implying evolution or renaming without explicit evidence or justification, which is not allowed."]

WRITER (revision 1)
Draft length: 659 words

Rule check: PASSED

Judge result:
  Score: 8/10
  Approved: False
  False checks: 1
  Issues: ["Use of unsupported strong claim 'supporting the reliability and governance of reported climate-related metrics' in External assurance and controls section."]

WRITER (revision 2)
Draft length: 600 words

Rule check: PASSED

Judge result:
  Score: 8/10
  Approved: False
  False checks: 1
  Issues: ["Unsupported narrative about committee name changes in Management Re

In [12]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":         result["status"],
        "final_score":    result["judge_result"].get("overall_score"),
        "revisions":      result["revision_count"],
        "approved":       result["judge_result"].get("approved"),
        "checklist":      result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")


FINAL JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 9,
  "specificity_score": 9,
  "hallucination_risk": "low",
  "approved": false,
  "checklist": {
    "all_six_subsections_present": true,
    "agenda_pct_correct": true,
    "four_distinct_decisions": true,
    "yoy_trends_present": true,
    "both_remuneration_figures": true,
    "s2_6b_process_complete": true,
    "no_unsupported_committee_evolution": false,
    "correct_ifrs_tags": true,
    "assurance_explained_safely": true,
    "no_limitation_disclaimer": true,
    "no_unsupported_strong_claims": true,
    "false_count": 1
  },
  "main_issues": [
    "Unsupported narrative about committee name changes in Management Responsibility subsection: 'The recorded management committee name was Group Sustainability Committee in 2022 and ESG Executive Committee in 2023.' This implies committee evolution/renaming without explicit evidence or justification."
  ],
  "required_fixes": [
    "Re